# nano-dsv4.1f — CPU tokenizer build

This notebook freezes the **32,768-token byte-BPE tokenizer** for `nano-dsv4.1f`.

It deliberately does **not** upload anything to GitHub. It:
1. clones the public nano repo so the special-token contract comes from source;
2. streams the pinned Hugging Face FineWeb `sample-10BT` corpus;
3. trains byte-level BPE on a bounded amount of streamed UTF-8 text;
4. validates the reserved DeepSeek V4.1-compatible protocol token IDs;
5. writes a manifest + SHA-256 hashes and a ZIP bundle under `/kaggle/working`.

No GitHub or Hugging Face secret is required for the default public-source workflow.

In [ ]:
import subprocess, sys
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "--upgrade",
    "datasets>=3.0", "tokenizers>=0.21",
], check=True)


In [ ]:
from __future__ import annotations

import hashlib
import importlib.metadata as md
import json
from pathlib import Path
import shutil
import subprocess
import sys
import time

REPO_URL = "https://github.com/xiayicheng3-code/nano-dsv4.1f.git"
REPO_REF = "main"

FINEWEB_REPO = "HuggingFaceFW/fineweb"
FINEWEB_CONFIG = "sample-10BT"
FINEWEB_REVISION = "9bb295ddab0e05d785b879661af7260fed5140fc"

SEED = 1701
SHUFFLE_BUFFER = 10_000
TARGET_UTF8_BYTES = 1_500_000_000
MAX_DOCUMENTS = None

VOCAB_SIZE = 32_768
MIN_FREQUENCY = 2

WORK = Path("/kaggle/working")
REPO_DIR = WORK / "nano-dsv4.1f"
OUTPUT_DIR = WORK / "nano-dsv41f-tokenizer"
ZIP_PATH = WORK / "nano-dsv41f-tokenizer.zip"

print({
    "target_GB": TARGET_UTF8_BYTES / 1e9,
    "fineweb_revision": FINEWEB_REVISION,
    "seed": SEED,
    "vocab_size": VOCAB_SIZE,
})

In [ ]:
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)],
    check=True,
)
nano_commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()

sys.path.insert(0, str(REPO_DIR / "src"))

from nano_dsv41f.chat_protocol import nano_v41_tokenizer_contract
from nano_dsv41f.tokenizer_training import train_byte_bpe_tokenizer

contract = nano_v41_tokenizer_contract(VOCAB_SIZE)
print("nano repo commit:", nano_commit)
print("reserved tokens:", len(contract.special_tokens))
print("first reserved IDs:", list(contract.token_to_id.items())[:6])

In [ ]:
from datasets import load_dataset

stream = load_dataset(
    FINEWEB_REPO,
    name=FINEWEB_CONFIG,
    split="train",
    streaming=True,
    revision=FINEWEB_REVISION,
)
stream = stream.shuffle(seed=SEED, buffer_size=SHUFFLE_BUFFER)

first = next(iter(stream))
print("fields:", sorted(first))
print("first text chars:", len(first["text"]))
print(first["text"][:300].replace("\n", " "))

In [ ]:
class BoundedFineWebTexts:
    """Single-pass text iterable that records exactly what the trainer consumed."""

    def __init__(self, dataset, *, target_bytes: int, max_documents: int | None = None):
        self.dataset = dataset
        self.target_bytes = int(target_bytes)
        self.max_documents = max_documents
        self.documents = 0
        self.utf8_bytes = 0
        self.characters = 0
        self.started_at = None
        self.finished_at = None

    def __iter__(self):
        self.started_at = time.time()
        for row in self.dataset:
            text = row.get("text")
            if not isinstance(text, str) or not text:
                continue

            encoded_bytes = len(text.encode("utf-8"))
            self.documents += 1
            self.utf8_bytes += encoded_bytes
            self.characters += len(text)
            yield text

            if self.utf8_bytes >= self.target_bytes:
                break
            if self.max_documents is not None and self.documents >= self.max_documents:
                break
        self.finished_at = time.time()

texts = BoundedFineWebTexts(
    stream,
    target_bytes=TARGET_UTF8_BYTES,
    max_documents=MAX_DOCUMENTS,
)

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

tokenizer_path = train_byte_bpe_tokenizer(
    texts,
    OUTPUT_DIR,
    contract=contract,
    min_frequency=MIN_FREQUENCY,
)

print({
    "tokenizer": str(tokenizer_path),
    "documents": texts.documents,
    "utf8_GB": round(texts.utf8_bytes / 1e9, 3),
    "characters": texts.characters,
    "elapsed_minutes": round((texts.finished_at - texts.started_at) / 60, 2),
})

In [ ]:
from tokenizers import Tokenizer

tok = Tokenizer.from_file(str(tokenizer_path))

assert tok.get_vocab_size(with_added_tokens=True) == VOCAB_SIZE
for token, expected_id in contract.token_to_id.items():
    actual = tok.token_to_id(token)
    assert actual == expected_id, (token, actual, expected_id)

samples = [
    "Hello, world!",
    "def f(x):\n    return x ** 2\n",
    "Unicode: 奶蛙 🐸 café naïve",
    "<｜User｜>hello<｜Assistant｜><think>test</think>",
]
for sample in samples:
    ids = tok.encode(sample).ids
    decoded = tok.decode(ids, skip_special_tokens=False)
    print({"text": sample[:40], "n_tokens": len(ids), "decoded": decoded[:80]})

print("special-token contract: PASS")

In [ ]:
manifest = {
    "format": "nano-dsv41f-tokenizer-build-v1",
    "nano_repo_url": REPO_URL,
    "nano_repo_commit": nano_commit,
    "source": {
        "dataset": FINEWEB_REPO,
        "config": FINEWEB_CONFIG,
        "revision": FINEWEB_REVISION,
        "streaming": True,
        "shuffle_seed": SEED,
        "shuffle_buffer": SHUFFLE_BUFFER,
    },
    "sample": {
        "target_utf8_bytes": TARGET_UTF8_BYTES,
        "actual_utf8_bytes": texts.utf8_bytes,
        "documents": texts.documents,
        "characters": texts.characters,
        "max_documents": MAX_DOCUMENTS,
    },
    "tokenizer": {
        "algorithm": "byte-level BPE",
        "vocab_size": VOCAB_SIZE,
        "min_frequency": MIN_FREQUENCY,
        "reserved_special_tokens": list(contract.special_tokens),
        "special_token_ids": contract.token_to_id,
    },
    "packages": {
        name: md.version(name)
        for name in ("datasets", "tokenizers", "huggingface-hub")
    },
}
(OUTPUT_DIR / "build_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

hashes = {
    p.name: sha256_file(p)
    for p in sorted(OUTPUT_DIR.iterdir())
    if p.is_file() and p.name != "SHA256SUMS.json"
}
(OUTPUT_DIR / "SHA256SUMS.json").write_text(
    json.dumps(hashes, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)

print(json.dumps(manifest, indent=2)[:4000])
print("hashes:", hashes)

In [ ]:
if ZIP_PATH.exists():
    ZIP_PATH.unlink()
shutil.make_archive(str(ZIP_PATH.with_suffix("")), "zip", OUTPUT_DIR)

print("bundle:", ZIP_PATH)
print("bundle bytes:", ZIP_PATH.stat().st_size)
print("contents:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(" ", p.name, p.stat().st_size)

## After the run

The Kaggle output bundle is `nano-dsv41f-tokenizer.zip`.

**No GitHub token is required.** Download or attach that ZIP in the ChatGPT conversation; the existing connected GitHub app can then commit the small tokenizer files into this repository.

If you later decide the notebook itself should push directly from Kaggle, *then* Kaggle would need a GitHub credential with write access (preferably a fine-grained token stored as a Kaggle Secret). The default notebook intentionally avoids that credential path.